In [1]:
"""
No-controls bivariate analysis of housing outcomes by race/ethnicity, using the
Howell-replication NYCHVS dataset.

This is the parallel of the "Correlative Results" sections in the R script
(Fixed_Assets__Neighborhood_Level.Rmd) and the no-controls columns of Howell
et al. (2023) Table 2. Each outcome is regressed on race alone -- no other
covariates -- separately for unsubsidized and subsidized renters.

For each outcome (unsafe conditions, housing cost, affordability):
  1. Weighted means by race (overall, unsubsidized, subsidized)
  2. Weighted bivariate OLS on race dummies with White as reference, run
     separately for unsubsidized and subsidized renters
  3. Top-5 / bottom-5 race rankings paralleling the R script's top-5 prints

Variance estimation:
  * Point estimates always use the full-sample weight FW.
  * If FW1..FW80 (housing-unit replicate weights) are present in the input CSV,
    standard errors are computed via the NYCHVS successive-difference replication
    (SDR) formula:  Var(theta) = (4/80) * sum_{r=1}^{80} (theta_r - theta_0)^2
    (Source: 2023 NYCHVS Sample Design, Weighting, and Error Estimation, Sec 5.)
  * Otherwise, falls back to statsmodels' i.i.d. WLS standard errors and prints
    a warning.

Notes:
  * Tiny race cells (AIAN_NHPI n=21, Other_2plus n=30) produce noisy estimates
    -- they are reported but flagged.
  * Records with missing values on the outcome or covariate are dropped
    pairwise per analysis.
"""


'\nNo-controls bivariate analysis of housing outcomes by race/ethnicity, using the\nHowell-replication NYCHVS dataset.\n\nThis is the parallel of the "Correlative Results" sections in the R script\n(Fixed_Assets__Neighborhood_Level.Rmd) and the no-controls columns of Howell\net al. (2023) Table 2. Each outcome is regressed on race alone -- no other\ncovariates -- separately for unsubsidized and subsidized renters.\n\nFor each outcome (unsafe conditions, housing cost, affordability):\n  1. Weighted means by race (overall, unsubsidized, subsidized)\n  2. Weighted bivariate OLS on race dummies with White as reference, run\n     separately for unsubsidized and subsidized renters\n  3. Top-5 / bottom-5 race rankings paralleling the R script\'s top-5 prints\n\nVariance estimation:\n  * Point estimates always use the full-sample weight FW.\n  * If FW1..FW80 (housing-unit replicate weights) are present in the input CSV,\n    standard errors are computed via the NYCHVS successive-difference rep

In [4]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import norm

In [5]:
INPUT_CSV = "../data/processed/howell_replication_nychvs.csv"
N_REPLICATES = 80                                       # NYCHVS uses 80 replicates
SDR_SCALE    = 4.0 / N_REPLICATES                       # NYCHVS SDR variance multiplier
REP_WEIGHT_COLS = [f"FW{i}" for i in range(1, N_REPLICATES + 1)]

In [6]:
# ---------------------------------------------------------------------------
# Load
# ---------------------------------------------------------------------------
df = pd.read_csv(INPUT_CSV)
print(f"Loaded {len(df):,} rows.\n")

# Detect replicate weights
HAS_REPLICATES = all(c in df.columns for c in REP_WEIGHT_COLS)
if HAS_REPLICATES:
    print(f"Detected replicate weights {REP_WEIGHT_COLS[0]}..{REP_WEIGHT_COLS[-1]}.")
    print("Standard errors will be computed via NYCHVS SDR formula "
          "(Var = (4/80) * sum (theta_r - theta_0)^2).\n")
else:
    print("Replicate weights FW1..FW80 not found in input file.")
    print("Falling back to i.i.d. WLS standard errors. These will UNDERSTATE "
          "true sampling uncertainty for a complex survey like NYCHVS -- treat "
          "p-values and CIs as approximate.\n")

# Outcome list and labels
OUTCOMES = {
    "UNSAFE_COND":    "Unsafe conditions (count, 0-6)",
    "HOUSING_COST":   "Monthly housing cost ($)",
    "AFFORDABILITY":  "Rent-to-income ratio",
}

# Race ordering: keep White first as reference; small cells last
RACE_ORDER = ["White", "Black", "Hispanic", "Asian", "AIAN_NHPI", "Other_2plus"]

# Subsidy split (Howell ran unsubsidized and subsidized separately in Table 2)
df["IS_SUBSIDIZED"] = df["SUBSIDY"].isin(["Public", "Voucher", "Other_subsidy"])

Loaded 3,540 rows.

Detected replicate weights FW1..FW80.
Standard errors will be computed via NYCHVS SDR formula (Var = (4/80) * sum (theta_r - theta_0)^2).



<positron-console-cell-6>:30: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`


In [7]:
# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def weighted_mean(x, w):
    """Weighted mean ignoring NaNs in x."""
    mask = ~np.isnan(x)
    if mask.sum() == 0:
        return np.nan
    return np.average(x[mask], weights=w[mask])


def weighted_summary_by_race(data, outcome, race_col="RACE", weight_col="FW"):
    """Weighted mean and unweighted N per race for one outcome."""
    rows = []
    for race in RACE_ORDER:
        sub = data[data[race_col] == race]
        x = sub[outcome].to_numpy(dtype=float)
        w = sub[weight_col].to_numpy(dtype=float)
        n = (~np.isnan(x)).sum()
        rows.append({
            "race": race,
            "n": int(n),
            "weighted_mean": weighted_mean(x, w),
        })
    return pd.DataFrame(rows)


def fit_wls(data, outcome, weight_col):
    """
    Weighted OLS: outcome ~ C(RACE) with White as reference, using the supplied
    weight column. Returns the fitted result and the analytic frame actually
    used (after dropping NAs on outcome).
    """
    cols = [outcome, "RACE", weight_col]
    sub = data[cols].dropna().copy()
    sub["RACE"] = pd.Categorical(sub["RACE"], categories=RACE_ORDER)
    X = pd.get_dummies(sub["RACE"], drop_first=True).astype(float)  # White dropped
    X = sm.add_constant(X, has_constant="add")
    y = sub[outcome].astype(float)
    w = sub[weight_col].astype(float)
    return sm.WLS(y, X, weights=w).fit(), sub


def bivariate_estimates(data, outcome):
    """
    Compute point estimates (using FW) plus standard errors.

    If replicate weights are present, SE comes from NYCHVS SDR:
        Var(theta_hat) = (4/80) * sum_{r=1}^{80} (theta_r - theta_0)^2
    where theta_0 uses FW and theta_r uses FW{r}. Per the NYCHVS docs, replicate
    1 equals the full sample weight by construction, so its squared difference
    is zero -- it is still included in the sum as the formula prescribes.

    Otherwise SE comes from statsmodels' i.i.d. WLS.

    Returns a DataFrame with one row per regression term.
    """
    point_model, used = fit_wls(data, outcome, weight_col="FW")
    point_coefs = point_model.params              # pandas Series indexed by term
    n_used      = len(used)
    r2          = point_model.rsquared

    if HAS_REPLICATES:
        # Re-fit with each replicate weight; collect coefficient vectors.
        # Note: each replicate fit uses the SAME records (those with non-missing
        # outcome). Replicate weights are non-negative; statsmodels handles 0s.
        rep_coef_list = []
        for col in REP_WEIGHT_COLS:
            rep_model, _ = fit_wls(data, outcome, weight_col=col)
            rep_coef_list.append(rep_model.params.reindex(point_coefs.index))
        rep_coefs = pd.DataFrame(rep_coef_list)              # 80 rows x k cols
        sq_diffs  = (rep_coefs - point_coefs) ** 2           # broadcast subtraction
        variances = SDR_SCALE * sq_diffs.sum(axis=0)         # one variance per term
        ses       = np.sqrt(variances)
        se_method = "SDR replicate"
    else:
        ses       = point_model.bse
        se_method = "WLS i.i.d."

    # t-stats and two-sided p-values using normal approximation. Survey
    # practitioners often use t with design df, but NYCHVS docs do not specify
    # a df, and at n>3000 the normal approx is essentially equivalent.
    z      = point_coefs / ses
    pvals  = 2.0 * (1.0 - norm.cdf(np.abs(z)))

    out = pd.DataFrame({
        "term": point_coefs.index,
        "coef": point_coefs.values,
        "se":   ses.values,
        "p":    pvals,
        "se_method": se_method,
        "n":    n_used,
        "r2":   r2,
    })
    return out

In [8]:
# ---------------------------------------------------------------------------
# 1. Sample sizes by race x subsidy status
# ---------------------------------------------------------------------------
print("=" * 70)
print("SAMPLE SIZES BY RACE x SUBSIDY STATUS")
print("=" * 70)
xt = pd.crosstab(df["RACE"].astype(pd.CategoricalDtype(RACE_ORDER, ordered=True)),
                 df["IS_SUBSIDIZED"].map({True: "Subsidized", False: "Unsubsidized"}),
                 margins=True)
print(xt)

# Weighted population shares by race
print("\nWeighted race composition (% of low-income renter HHs in NYC):")
totw = df["FW"].sum()
for race in RACE_ORDER:
    s = df.loc[df["RACE"] == race, "FW"].sum() / totw * 100
    print(f"  {race:<12s}: {s:5.2f}%")

SAMPLE SIZES BY RACE x SUBSIDY STATUS
IS_SUBSIDIZED  Subsidized  Unsubsidized   All
RACE                                         
White                 113           667   780
Black                 223           770   993
Hispanic              290          1055  1345
Asian                  37           334   371
AIAN_NHPI               5            16    21
Other_2plus             4            26    30
All                   672          2868  3540

Weighted race composition (% of low-income renter HHs in NYC):
  White       : 22.98%
  Black       : 27.33%
  Hispanic    : 36.03%
  Asian       : 12.04%
  AIAN_NHPI   :  0.80%
  Other_2plus :  0.82%


In [9]:
# ---------------------------------------------------------------------------
# 2. Weighted means by race (overall, unsub, sub)
# ---------------------------------------------------------------------------
print("\n" + "=" * 70)
print("WEIGHTED MEANS BY RACE")
print("=" * 70)

for outcome, label in OUTCOMES.items():
    print(f"\n--- {label} ({outcome}) ---")

    overall = weighted_summary_by_race(df, outcome).rename(
        columns={"n": "n_all", "weighted_mean": "mean_all"})
    unsub = weighted_summary_by_race(df[~df["IS_SUBSIDIZED"]], outcome).rename(
        columns={"n": "n_unsub", "weighted_mean": "mean_unsub"})
    sub = weighted_summary_by_race(df[df["IS_SUBSIDIZED"]], outcome).rename(
        columns={"n": "n_sub", "weighted_mean": "mean_sub"})

    merged = overall.merge(unsub, on="race").merge(sub, on="race")
    fmt = "{:.3f}" if outcome in ("AFFORDABILITY", "UNSAFE_COND") else "{:,.0f}"
    for c in ["mean_all", "mean_unsub", "mean_sub"]:
        merged[c] = merged[c].map(lambda v: fmt.format(v) if pd.notna(v) else "")
    print(merged.to_string(index=False))


WEIGHTED MEANS BY RACE

--- Unsafe conditions (count, 0-6) (UNSAFE_COND) ---
       race  n_all mean_all  n_unsub mean_unsub  n_sub mean_sub
      White    668    1.118      561      1.090    107    1.310
      Black    925    1.859      715      1.841    210    1.924
   Hispanic   1229    1.708      954      1.698    275    1.742
      Asian    333    0.973      297      0.955     36    1.150
  AIAN_NHPI     19    1.547       14      1.295      5    2.456
Other_2plus     27    2.253       23      2.034      4    4.064

--- Monthly housing cost ($) (HOUSING_COST) ---
       race  n_all mean_all  n_unsub mean_unsub  n_sub mean_sub
      White    780    1,715      667      1,764    113    1,345
      Black    993    1,211      770      1,215    223    1,197
   Hispanic   1345    1,345     1055      1,350    290    1,323
      Asian    371    1,455      334      1,501     37      989
  AIAN_NHPI     21    1,538       16      1,623      5    1,194
Other_2plus     30    1,508       26     

In [10]:
# ---------------------------------------------------------------------------
# 3. Bivariate WLS regressions: outcome ~ race (no controls)
# ---------------------------------------------------------------------------
print("\n" + "=" * 70)
print("BIVARIATE WLS: outcome ~ RACE (White = reference, no other controls)")
print(f"SE method: {'NYCHVS SDR replicate' if HAS_REPLICATES else 'i.i.d. WLS (approximate)'}")
print("=" * 70)

all_results = []
for outcome, label in OUTCOMES.items():
    print(f"\n--- {label} ({outcome}) ---")
    for sub_label, sub_data in [
        ("Unsubsidized", df[~df["IS_SUBSIDIZED"]]),
        ("Subsidized",   df[df["IS_SUBSIDIZED"]]),
        ("All renters",  df),
    ]:
        tab = bivariate_estimates(sub_data, outcome)
        tab["outcome"] = outcome
        tab["sample"]  = sub_label
        all_results.append(tab)

        n  = int(tab["n"].iloc[0])
        r2 = float(tab["r2"].iloc[0])
        print(f"\n  [{sub_label}, n={n:,}, R^2={r2:.4f}]")
        for _, row in tab.iterrows():
            term = row["term"] if row["term"] != "const" else "(Intercept = White mean)"
            sig = ("***" if row["p"] < .001 else
                   "**"  if row["p"] < .01  else
                   "*"   if row["p"] < .05  else "")
            print(f"    {term:<28s}  {row['coef']:>10.3f}  (SE {row['se']:.3f}) {sig}")

# Save full results table
results_df = pd.concat(all_results, ignore_index=True)
results_df.to_csv("../data/output/no_controls_regression_results.csv", index=False)
print("\nSaved regression results to data/output/no_controls_regression_results.csv")


BIVARIATE WLS: outcome ~ RACE (White = reference, no other controls)
SE method: NYCHVS SDR replicate

--- Unsafe conditions (count, 0-6) (UNSAFE_COND) ---

  [Unsubsidized, n=2,564, R^2=0.0496]
    (Intercept = White mean)           1.090  (SE 0.066) ***
    Black                              0.751  (SE 0.099) ***
    Hispanic                           0.608  (SE 0.084) ***
    Asian                             -0.135  (SE 0.085) 
    AIAN_NHPI                          0.205  (SE 0.498) 
    Other_2plus                        0.945  (SE 0.442) *

  [Subsidized, n=637, R^2=0.0334]
    (Intercept = White mean)           1.310  (SE 0.130) ***
    Black                              0.614  (SE 0.205) **
    Hispanic                           0.432  (SE 0.165) **
    Asian                             -0.161  (SE 0.316) 
    AIAN_NHPI                          1.146  (SE 0.390) **
    Other_2plus                        2.753  (SE 1.329) *

  [All renters, n=3,201, R^2=0.0465]
    (Intercept =

OSError: Cannot save file into a non-existent directory: '../data/output'

In [ ]:
# ---------------------------------------------------------------------------
# 4. Race rankings (parallels R top-5 prints)
# ---------------------------------------------------------------------------
print("\n" + "=" * 70)
print("RACE GROUPS RANKED BY EACH OUTCOME (overall sample)")
print("=" * 70)

for outcome, label in OUTCOMES.items():
    print(f"\n--- {label} (all eligible renters, weighted means) ---")
    rank = weighted_summary_by_race(df, outcome).sort_values("weighted_mean")
    direction = {
        "UNSAFE_COND":   "Lower = better unit quality",
        "HOUSING_COST":  "Lower = cheaper rent",
        "AFFORDABILITY": "Lower = less rent burden",
    }[outcome]
    print(f"  ({direction})")
    for _, r in rank.iterrows():
        if outcome in ("UNSAFE_COND", "AFFORDABILITY"):
            print(f"    {r['race']:<12s}  mean = {r['weighted_mean']:.3f}   (n={r['n']})")
        else:
            print(f"    {r['race']:<12s}  mean = ${r['weighted_mean']:,.0f}   (n={r['n']})")


RACE GROUPS RANKED BY EACH OUTCOME (overall sample)

--- Unsafe conditions (count, 0-6) (all eligible renters, weighted means) ---
  (Lower = better unit quality)
    Asian         mean = 0.968   (n=338)
    White         mean = 1.113   (n=679)
    AIAN_NHPI     mean = 1.547   (n=19)
    Hispanic      mean = 1.700   (n=1235)
    Black         mean = 1.854   (n=930)
    Other_2plus   mean = 2.253   (n=27)

--- Monthly housing cost ($) (all eligible renters, weighted means) ---
  (Lower = cheaper rent)
    Black         mean = $1,216   (n=999)
    Hispanic      mean = $1,359   (n=1354)
    Other_2plus   mean = $1,508   (n=30)
    AIAN_NHPI     mean = $1,538   (n=21)
    Asian         mean = $1,544   (n=379)
    White         mean = $1,745   (n=794)

--- Rent-to-income ratio (all eligible renters, weighted means) ---
  (Lower = less rent burden)
    Black         mean = 0.416   (n=999)
    Hispanic      mean = 0.428   (n=1354)
    Other_2plus   mean = 0.446   (n=30)
    AIAN_NHPI     mea

In [ ]:
# ---------------------------------------------------------------------------
# 5. Subsidized-vs-unsubsidized gap by race (Howell's central comparison)
# ---------------------------------------------------------------------------
print("\n" + "=" * 70)
print("SUBSIDIZED vs UNSUBSIDIZED: WHITE-vs-OTHER GAPS")
print("=" * 70)
print("\nFrom the bivariate regressions: race coefficients (gap relative to White).")
print("Howell's argument: gaps are LARGER among subsidized renters than unsubsidized.\n")

# Reuse the already-fit results stored in `results_df` for the relevant subsets.
gap_df = (
    results_df[
        results_df["sample"].isin(["Unsubsidized", "Subsidized"])
        & (results_df["term"] != "const")
    ]
    .rename(columns={"term": "race", "coef": "gap_vs_White"})
    [["outcome", "sample", "race", "gap_vs_White", "se", "p"]]
    .reset_index(drop=True)
)

piv = gap_df.pivot_table(index="race", columns=["outcome", "sample"], values="gap_vs_White")
piv = piv.reindex([r for r in RACE_ORDER if r != "White"])
print(piv.round(3).to_string())

gap_df.to_csv("../data/output/no_controls_gap_by_race_subsidy.csv", index=False)
print("\nSaved gap table to ../data/outputs/no_controls_gap_by_race_subsidy.csv")

print("\nDone.")



SUBSIDIZED vs UNSUBSIDIZED: WHITE-vs-OTHER GAPS

From the bivariate regressions: race coefficients (gap relative to White).
Howell's argument: gaps are LARGER among subsidized renters than unsubsidized.

outcome     AFFORDABILITY              HOUSING_COST              UNSAFE_COND             
sample         Subsidized Unsubsidized   Subsidized Unsubsidized  Subsidized Unsubsidized
race                                                                                     
Black              -0.036       -0.068     -290.128     -463.269       0.690        0.628
Hispanic           -0.019       -0.057     -172.167     -319.290       0.648        0.422
Asian               0.006        0.011     -365.827     -178.792       0.048       -0.173
AIAN_NHPI          -0.003       -0.003     -153.071     -150.669       0.890        0.248
Other_2plus        -0.052       -0.014     -451.705     -138.208       2.027        0.828

Saved gap table to ../data/outputs/no_controls_gap_by_race_subsidy.csv

Do